# **RoPE - Rotary Position Embedding**

## Введение

Позиционное кодирование в последнее время показало свою эффективность в архитектуре transformer. Оно позволяет
осуществлять контроль зависимостей между элементами, расположенными в разных частях последовательности. В этом эссе мы рассмотрим новый метод, получивший название Rotary (поворотный).

Предлагаемый RoPE кодирует абсолютное положение с помощью матрицы вращения и в то же время включает явную зависимость от относительного положения при подсчёте self-attenrion.

Примечательно, что RoPE обладает рядом преимуществ, таких как гибкость в отношении длины последовательности, ослабление зависимости между токенами с увеличением относительных расстояний и возможность оснащения линейного self-attention
кодированием относительного положения.

## Задача

Оптимизация модели transformer с помощью нового метода позиционного кодирования, которое бы учитывало и абсолютные положения слов/токенов во входящей последовательности, и относительные расстояния между ними.

Пусть $S_N = ${$w_i$}$_{i=1}$$^N$ - последовательность из N входных токенов. Соответствующий эмбеддинг токена $w_i$ обозначается как $E_N$ = {$x_i$}$_{i=1}$$^N$, где $x_i ∈ R^d$ - это d-мерный вектор-эмбеддинг токена $w_i$ без позиционной информации. В self-attention сначала  включаем информацию о местоположении эмбеддинго, а затем преобразовываем их в запросы, ключи и значения (queries, keys, values).

- $q_m = f_q(x_m, m)$
- $k_n = f_k(x_n, n)$
- $v_n = f_v(x_n, n)$

где $q_m, k_n$ и $v_n$ включают m-ю и n-ю позиции с помощью функций $f_q, f_k$ и $f_v$ соответственно. Затем значения query и key
используются для вычисления весов внимания (attention weights), а выходные данные вычисляются как взвешенная сумма со значениями values.

$a_{m,n}$ = $\frac{exp(\frac{q^T_mk_n}{\sqrt(d)})}{∑_{j=1}^N exp(\frac{q^T_mk_j}{\sqrt(d)})}$


$output_m$ = $∑_{j=1}^N a_{m,n}v_n$

Существующие подходы к позиционному кодированию в основном сосредоточены на выборе подходящих функций $f_q, f_k, f_v$.

## Абсолютное позиционное кодирование

Типичный выбор функций f следующий:

$f_{t:t∈{q,k,v}}(x_i, i) := W_{t:t∈{q,k,v}}(x_i + p_i)$,

где $p_i ∈ R^d$ - это d-мерноый вектор, зависящий от позиции токена $x_i$. Ранее было предложено их считать с помощью синусоидальных функций:

- $p_{i,2t} = sin(k/10000^{2t/d})$
- $p_{i,2t+1} = cos(k/10000^{2t/d})$

в котором $p_{i,2t}$ - это 2t-й элемент d-мерного вектора $p_i$. Однако вместо прямого добавления местоположения
к контекстному представлению RoPE предлагает включить информацию об относительном местоположении путем умножения на
синусоидальные функции.

## Относительное позиционное кодирование

В статье "Peter Shaw, Jakob Uszkoreit, and Ashish Vaswani. Self-attention with relative position representations. In NAACL-HLT,
2018" предлагается следующее представление для нахождения функций $f_{i=q,k,v}$:


- $f_q(x_m) := W_qx_m$
- $f_k(x_n, n) := W_k(x_n + p˜_r^k)$
- $f_v(x_n, n) := W_v(x_n + p˜_r^v)$

где $p˜_r^k, p˜_r^v ∈ R^d$ - обучаемые относительные расстояния-эмбеддинги.


Чтобы включить информацию об относительном местоположении, нам требуется, чтобы скалярное произведение запроса
$q_m$ и ключа $k_n$ было представимо функцией g, которая принимает только эмбеддинги слов $x_m$, $x_n$ и их относительное
расстояние (m−n) в качестве входных переменных:

$⟨f_q(x_m, m), f_k(x_n, n)⟩ = g(x_m, x_n, m−n)$.


Конечная цель состоит в том, чтобы найти эквивалентный механизм кодирования для нахождения функций $f_q(x_m, m)$ и $f_k(x_n, n)$ в
соответствии с вышеупомянутым соотношением.

## Rotary position embedding

В общем виде представление $f_q$ и $f_k$ выглядит следующим образом:

$f_{q,v}(x_m, m) := R_{\theta,m}^d W_{q,v}(x_m)$

$R_{\theta,m}^d$ =
\begin{pmatrix}
cos(m\theta_1) & -sin(m\theta_1) & 0 & 0 & ... & 0 & 0 \\
sin(m\theta_1) & cos(m\theta_1) & 0 & 0 & ... & 0 & 0 \\
0 & 0 & cos(m\theta_2) & -sin(m\theta_2) &  ... & 0 & 0 \\
0 & 0 & sin(m\theta_2) & cos(m\theta_2) &  ... & 0 & 0 \\
...\\
0 & 0 & 0 & 0 & ... & cos(m\theta_{d/2}) & -sin(m\theta_{d/2})  \\
0 & 0 & 0 & 0 & ... & sin(m\theta_{d/2}) & cos(m\theta_{d/2})
\end{pmatrix}

## Свойства RoPE

### Долговременное уменьшение (long-term decay)

Устанавливаем $\theta_i = 10000^{-2i/d}$. Данное значение позволяет сохранять долгосрочные зависимости: скалярное произведение будет уменьшаться в то время, как относительное расстояние - увеличиваться. Что довольно логично: пара токенов с довольно большим расстоянием относительно друг друга должна иметь меньшее влияние друг на друга.

### RoPE с линейным self-attention

Вспомним, что исходный self-attention должен
вычислять скалярное произведение query и key для каждой пары токенов, что имеет квадратичную сложность $O(N^2)$.

Поскольку RoPE вносит информацию о местоположении путем поворота, что сохраняет норму скрытых слоёв неизменной, мы можем объединить RoPE с линейным вниманием, умножив матрицу поворота на
выходные данные неотрицательных функций:

$Attention(Q, K, V)_m$ = $\frac{∑_{n=1}^N (R_{\theta,m}^d\phi(q_m))^T(R_{\theta,m}^d\psi(k_n))v_n}{∑_{n=1}^N \phi(q_m)^T\psi(k_n)}$


In [1]:
import torch
from torch import nn as nn
from torch import optim as optim
import torch.utils.data as data
import math
import copy

In [2]:
class RotaryEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        inv_freq = 1. / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)

    def forward(self, x, seq_len):
        """
        x: [batch, num_heads, seq_len, dim_head]
        """
        t = torch.arange(seq_len, device=x.device).type_as(self.inv_freq)
        freqs = torch.einsum('i,j->ij', t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        cos = emb.cos().unsqueeze(0).unsqueeze(0)
        sin = emb.sin().unsqueeze(0).unsqueeze(0)
        return cos, sin

def apply_rotary_pos_emb(q, k, cos, sin):
    """
    q, k: [batch, num_heads, seq_len, dim_head]
    cos, sin: [1, 1, seq_len, dim_head]
    """
    def rotate_half(x):
        # Разбиваем на пары
        x = x.unflatten(-1, (-1, 2))
        x = torch.stack([-x[..., 1], x[..., 0]], dim=-1)
        return x.flatten(-2)

    q_rotary = (q * cos) + (rotate_half(q) * sin)
    k_rotary = (k * cos) + (rotate_half(k) * sin)

    return q_rotary, k_rotary

## В силу разреженности матрицы $R_{\theta,m}^d$
мы можем посчитать умножение на неё более эффективным способом:

$R_{\theta,m}^d x$ =  
\begin{pmatrix}x_1 \\x_2 \\x_3 \\x_4 \\...\\x_{d-1}  \\x_d \end{pmatrix} ⊙
\begin{pmatrix}
cos(m\theta_1) \\
cos(m\theta_1) \\
cos(m\theta_2) \\
cos(m\theta_2) \\
...\\
cos(m\theta_{d/2}) \\
cos(m\theta_{d/2})
\end{pmatrix} ⊕
\begin{pmatrix}
-x_2 \\
x_1 \\
-x_4 \\
x_3 \\
...\\
-x_d  \\
x_{d-1}
\end{pmatrix} ⊙
\begin{pmatrix}
sin(m\theta_1) \\
sin(m\theta_1) \\
sin(m\theta_2) \\
sin(m\theta_2) \\
...\\
sin(m\theta_{d/2}) \\
sin(m\theta_{d/2})
\end{pmatrix}



In [3]:
class MultiHeadAttentionWithRoPE(nn.Module):
    def __init__(self, dim_model, num_heads, rope = False, is_self_attn = True):
        super(MultiHeadAttentionWithRoPE, self).__init__()
        assert dim_model % num_heads == 0, "dim_model must be divisible by num_heads"
        self.rope = rope
        self.is_self_attn = is_self_attn
        self.dim_model = dim_model
        self.num_heads = num_heads
        self.dim_head = dim_model // num_heads
        self.W_q = nn.Linear(dim_model, dim_model)
        self.W_k = nn.Linear(dim_model, dim_model)
        self.W_v = nn.Linear(dim_model, dim_model)
        self.W_output = nn.Linear(dim_model, dim_model)
        if self.rope:
          self.rotary_emb = RotaryEmbedding(self.dim_head)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.dim_head)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_probs = torch.softmax(attn_scores, dim=-1)
        output = torch.matmul(attn_probs, V)
        return output

    def split_heads(self, x):
        batch_size, seq_length, dim_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.dim_head).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, dim_head = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.dim_model)

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        if self.rope and self.is_self_attn:
          batch_size, num_heads, seq_length, dim_head = Q.size()
          cos, sin = self.rotary_emb(Q, seq_length)
          Q_rotary, K_rotary = apply_rotary_pos_emb(Q, K, cos, sin)
        else:
          Q_rotary = Q
          K_rotary = K
        attn_output = self.scaled_dot_product_attention(Q_rotary, K_rotary, V, mask)
        output = self.W_output(self.combine_heads(attn_output))
        return output

In [4]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, dim_model, dim_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(dim_model, dim_ff)
        self.fc2 = nn.Linear(dim_ff, dim_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, dim_model, max_seq_length):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_seq_length, dim_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim_model, 2).float() * -(math.log(10000.0) / dim_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [6]:
class EncoderLayer(nn.Module):
    def __init__(self, dim_model, num_heads, dim_ff, dropout, rope):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttentionWithRoPE(dim_model, num_heads, rope)
        self.feed_forward = PositionWiseFeedForward(dim_model, dim_ff)
        self.norm1 = nn.LayerNorm(dim_model)
        self.norm2 = nn.LayerNorm(dim_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

In [7]:
class DecoderLayer(nn.Module):
    def __init__(self, dim_model, num_heads, dim_ff, dropout, rope):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttentionWithRoPE(dim_model, num_heads, rope, is_self_attn = True)
        self.cross_attn = MultiHeadAttentionWithRoPE(dim_model, num_heads, rope, is_self_attn = False)
        self.feed_forward = PositionWiseFeedForward(dim_model, dim_ff)
        self.norm1 = nn.LayerNorm(dim_model)
        self.norm2 = nn.LayerNorm(dim_model)
        self.norm3 = nn.LayerNorm(dim_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask, tgt_mask):
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))
        attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        return x

In [8]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, dim_model, num_heads, num_layers, dim_ff, max_seq_length, dropout, rope):
        super(Transformer, self).__init__()
        self.encoder_embedding = nn.Embedding(src_vocab_size, dim_model)
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, dim_model)
        if not rope:
          self.positional_encoding = PositionalEncoding(dim_model, max_seq_length)

        self.encoder_layers = nn.ModuleList([EncoderLayer(dim_model, num_heads, dim_ff, dropout, rope) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(dim_model, num_heads, dim_ff, dropout, rope) for _ in range(num_layers)])

        self.fc = nn.Linear(dim_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, src, tgt):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3)
        seq_length = tgt.size(1)
        nopeak_mask = (1 - torch.triu(torch.ones(1, seq_length, seq_length), diagonal=1)).bool()
        tgt_mask = tgt_mask & nopeak_mask
        return src_mask, tgt_mask

    def forward(self, src, tgt, rope):
        src_mask, tgt_mask = self.generate_mask(src, tgt)
        if not rope:
          src_embedded = self.dropout(self.positional_encoding(self.encoder_embedding(src)))
          tgt_embedded = self.dropout(self.positional_encoding(self.decoder_embedding(tgt)))
        else:
          src_embedded = self.dropout(self.encoder_embedding(src))
          tgt_embedded = self.dropout(self.decoder_embedding(tgt))

        enc_output = src_embedded
        for enc_layer in self.encoder_layers:
            enc_output = enc_layer(enc_output, src_mask)

        dec_output = tgt_embedded
        for dec_layer in self.decoder_layers:
            dec_output = dec_layer(dec_output, enc_output, src_mask, tgt_mask)

        output = self.fc(dec_output)
        return output

In [9]:
src_vocab_size = 2000
tgt_vocab_size = 2000
dim_model = 512
num_heads = 8
num_layers = 6
dim_ff = 2048
max_seq_length = 100
batch_size = 64
dropout = 0.1

# Generate random sample data
src_data = torch.randint(1, src_vocab_size, (batch_size, max_seq_length))
tgt_data = torch.randint(1, tgt_vocab_size, (batch_size, max_seq_length))

In [11]:
transformer = Transformer(src_vocab_size, tgt_vocab_size, dim_model, num_heads, num_layers, dim_ff, max_seq_length, dropout, rope = False)
transformer_with_rope = Transformer(src_vocab_size, tgt_vocab_size, dim_model, num_heads, num_layers, dim_ff, max_seq_length, dropout, rope = True)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(transformer.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)
optimizer_rope = optim.Adam(transformer_with_rope.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

transformer.train()
transformer_with_rope.train()

for epoch in range(100):
    optimizer.zero_grad()
    optimizer_rope.zero_grad()
    output = transformer(src_data, tgt_data[:, :-1], rope = False)
    # output_rope = transformer_with_rope(src_data, tgt_data[:, :-1], rope = True)
    loss = criterion(output.contiguous().view(-1, tgt_vocab_size), tgt_data[:, 1:].contiguous().view(-1))
    # loss_rope = criterion(output_rope.contiguous().view(-1, tgt_vocab_size), tgt_data[:, 1:].contiguous().view(-1))
    loss.backward()
    # loss_rope.backward()
    optimizer.step()
    # optimizer_rope.step()
    # print(f"Epoch: {epoch+1}, Loss without RoPE: {loss.item()}, Loss with RoPE: {loss_rope.item()}")
    print(f"Epoch: {epoch+1}, Loss without RoPE: {loss.item()}")

Epoch: 1, Loss without RoPE: 7.781046390533447
Epoch: 2, Loss without RoPE: 7.654652118682861
Epoch: 3, Loss without RoPE: 7.5824761390686035
Epoch: 4, Loss without RoPE: 7.540801048278809
Epoch: 5, Loss without RoPE: 7.500471115112305
Epoch: 6, Loss without RoPE: 7.454583168029785
Epoch: 7, Loss without RoPE: 7.39271354675293
Epoch: 8, Loss without RoPE: 7.323623180389404
Epoch: 9, Loss without RoPE: 7.251718044281006
Epoch: 10, Loss without RoPE: 7.180927753448486
Epoch: 11, Loss without RoPE: 7.1070685386657715
Epoch: 12, Loss without RoPE: 7.038136005401611
Epoch: 13, Loss without RoPE: 6.967462539672852
Epoch: 14, Loss without RoPE: 6.889951229095459
Epoch: 15, Loss without RoPE: 6.817123889923096
Epoch: 16, Loss without RoPE: 6.735479831695557
Epoch: 17, Loss without RoPE: 6.661789894104004
Epoch: 18, Loss without RoPE: 6.588717460632324
Epoch: 19, Loss without RoPE: 6.511324405670166
Epoch: 20, Loss without RoPE: 6.444664001464844
Epoch: 21, Loss without RoPE: 6.3686604499816895

In [10]:
transformer_with_rope = Transformer(src_vocab_size, tgt_vocab_size, dim_model, num_heads, num_layers, dim_ff, max_seq_length, dropout, rope = True)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer_rope = optim.Adam(transformer_with_rope.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

transformer_with_rope.train()

for epoch in range(100):
    optimizer_rope.zero_grad()
    output_rope = transformer_with_rope(src_data, tgt_data[:, :-1], rope = True)
    loss_rope = criterion(output_rope.contiguous().view(-1, tgt_vocab_size), tgt_data[:, 1:].contiguous().view(-1))
    loss_rope.backward()
    optimizer_rope.step()
    # print(f"Epoch: {epoch+1}, Loss without RoPE: {loss.item()}, Loss with RoPE: {loss_rope.item()}")
    print(f"Epoch: {epoch+1}, Loss with RoPE: {loss_rope.item()}")

Epoch: 1, Loss with RoPE: 7.752316951751709
Epoch: 2, Loss with RoPE: 7.6215291023254395
Epoch: 3, Loss with RoPE: 7.545405864715576
Epoch: 4, Loss with RoPE: 7.486445426940918
Epoch: 5, Loss with RoPE: 7.421816349029541
Epoch: 6, Loss with RoPE: 7.343212604522705
Epoch: 7, Loss with RoPE: 7.256879806518555
Epoch: 8, Loss with RoPE: 7.175484657287598
Epoch: 9, Loss with RoPE: 7.090401649475098
Epoch: 10, Loss with RoPE: 7.000770092010498
Epoch: 11, Loss with RoPE: 6.924987316131592
Epoch: 12, Loss with RoPE: 6.832637310028076
Epoch: 13, Loss with RoPE: 6.7440185546875
Epoch: 14, Loss with RoPE: 6.655285358428955
Epoch: 15, Loss with RoPE: 6.573821067810059
Epoch: 16, Loss with RoPE: 6.48745059967041
Epoch: 17, Loss with RoPE: 6.406298637390137
Epoch: 18, Loss with RoPE: 6.335564613342285
Epoch: 19, Loss with RoPE: 6.264351844787598
Epoch: 20, Loss with RoPE: 6.162635803222656
Epoch: 21, Loss with RoPE: 6.102057456970215
Epoch: 22, Loss with RoPE: 6.021766185760498
Epoch: 23, Loss with 

In [34]:
from torch.utils.tensorboard import SummaryWriter
import os
from datetime import datetime

log_dir = "runs/transformer"
log_dir_rope = "runs/transformer_rope"
os.makedirs(log_dir, exist_ok=True)
os.makedirs(log_dir_rope, exist_ok=True)

writer = SummaryWriter(log_dir=log_dir)
writer_ROPE = SummaryWriter(log_dir=log_dir_rope)

val_ratio = 0.1  # 10% данных для валидации
src_vocab_size = 2000
tgt_vocab_size = 2000
dim_model = 512
num_heads = 8
num_layers = 6
dim_ff = 1024
max_seq_length = 100
batch_size = 32
dropout = 0.1

# src_data = torch.randint(1, src_vocab_size, (batch_size, max_seq_length))
# tgt_data = torch.randint(1, tgt_vocab_size, (batch_size, max_seq_length))

total_samples = 320
src_data_all = torch.randint(1, src_vocab_size, (total_samples, max_seq_length))
tgt_data_all = torch.randint(1, tgt_vocab_size, (total_samples, max_seq_length))

val_size = int(total_samples * val_ratio)
train_size = total_samples - val_size

indices = torch.randperm(total_samples)
train_indices = indices[:train_size]
val_indices = indices[train_size:]

src_train, tgt_train = src_data_all[train_indices], tgt_data_all[train_indices]
src_val, tgt_val = src_data_all[val_indices], tgt_data_all[val_indices]

print(f"Train: {train_size} samples, Val: {val_size} samples")

def validate(model, src_val, tgt_val, criterion, batch_size=64, device='cuda'):
    model.eval()

    total_loss = 0
    # total_tokens = 0

    with torch.no_grad():
        for i in range(0, len(src_val), batch_size):
            src_batch = src_val[i:i+batch_size].to(device)
            tgt_batch = tgt_val[i:i+batch_size].to(device)

            output = model(src_batch, tgt_batch[:, :-1], rope=True)

            loss = criterion(
                output.contiguous().view(-1, tgt_vocab_size),
                tgt_batch[:, 1:].contiguous().view(-1)
            )

            batch_tokens = (tgt_batch[:, 1:] != 0).sum().item()
            total_loss += loss.item() // batch_tokens
            # total_tokens += batch_tokens

    model.train()

    return total_loss

Train: 288 samples, Val: 32 samples


In [32]:
num_epochs = 50
lr = 0.0001
val_every = 5

transformer_with_rope = Transformer(src_vocab_size, tgt_vocab_size, dim_model, num_heads, num_layers, dim_ff, max_seq_length, dropout, rope = True)
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer_rope = optim.Adam(
    transformer_with_rope.parameters(),
    lr=lr,
    betas=(0.9, 0.98),
    eps=1e-9
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transformer_with_rope.to(device)

for epoch in range(num_epochs):
    transformer_with_rope.train()
    optimizer_rope.zero_grad()

    epoch_loss = 0
    n_batches = 0

    for i in range(0, len(src_train), batch_size):
        src_batch = src_train[i:i+batch_size].to(device)
        tgt_batch = tgt_train[i:i+batch_size].to(device)

        output = transformer_with_rope(src_batch, tgt_batch[:, :-1], rope=True)

        loss = criterion(
            output.contiguous().view(-1, tgt_vocab_size),
            tgt_batch[:, 1:].contiguous().view(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(transformer_with_rope.parameters(), max_norm=1.0)
        optimizer_rope.step()
        optimizer_rope.zero_grad()

        epoch_loss += loss.item()
        n_batches += 1

    train_loss = epoch_loss / n_batches

    if (epoch + 1) % val_every == 0 or epoch == num_epochs - 1:
        val_loss = validate(
            transformer_with_rope, src_val, tgt_val,
            criterion, batch_size=batch_size, device=device
        )

        writer.add_scalar('Loss/Train', train_loss, epoch)
        writer.add_scalar('Loss/Val', val_loss, epoch)

        print(f"Epoch {epoch+1:3d} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} ")
    else:
        writer.add_scalar('Loss/Train', train_loss, epoch)
        print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} ")

writer.close()

Epoch   1 | Train Loss: 7.7298 
Epoch   2 | Train Loss: 7.6185 
Epoch   3 | Train Loss: 7.5771 
Epoch   4 | Train Loss: 7.5403 
Epoch   5 | Train Loss: 7.4832 | Val Loss: 24.0000 
Epoch   6 | Train Loss: 7.3748 
Epoch   7 | Train Loss: 7.2582 
Epoch   8 | Train Loss: 7.1276 
Epoch   9 | Train Loss: 7.0006 
Epoch  10 | Train Loss: 6.8759 | Val Loss: 24.0000 
Epoch  11 | Train Loss: 6.7577 
Epoch  12 | Train Loss: 6.6491 
Epoch  13 | Train Loss: 6.5464 
Epoch  14 | Train Loss: 6.4473 
Epoch  15 | Train Loss: 6.3526 | Val Loss: 24.0000 
Epoch  16 | Train Loss: 6.2612 
Epoch  17 | Train Loss: 6.1776 
Epoch  18 | Train Loss: 6.0902 
Epoch  19 | Train Loss: 6.0099 
Epoch  20 | Train Loss: 5.9297 | Val Loss: 24.0000 
Epoch  21 | Train Loss: 5.8552 
Epoch  22 | Train Loss: 5.7759 


KeyboardInterrupt: 

In [35]:
num_epochs = 50
lr = 0.0001
val_every = 5

transformer_with_rope = Transformer(src_vocab_size, tgt_vocab_size, dim_model, num_heads, num_layers, dim_ff, max_seq_length, dropout, rope = True)
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer_rope = optim.Adam(
    transformer_with_rope.parameters(),
    lr=lr,
    betas=(0.9, 0.98),
    eps=1e-9
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transformer_with_rope.to(device)

for epoch in range(num_epochs):
    transformer_with_rope.train()
    optimizer_rope.zero_grad()

    epoch_loss = 0
    n_batches = 0

    for i in range(0, len(src_train), batch_size):
        src_batch = src_train[i:i+batch_size].to(device)
        tgt_batch = tgt_train[i:i+batch_size].to(device)

        output = transformer_with_rope(src_batch, tgt_batch[:, :-1], rope=True)

        loss = criterion(
            output.contiguous().view(-1, tgt_vocab_size),
            tgt_batch[:, 1:].contiguous().view(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(transformer_with_rope.parameters(), max_norm=1.0)
        optimizer_rope.step()
        optimizer_rope.zero_grad()

        epoch_loss += loss.item()
        n_batches += 1

    train_loss = epoch_loss / n_batches

    if (epoch + 1) % val_every == 0 or epoch == num_epochs - 1:
        val_loss = validate(
            transformer_with_rope, src_val, tgt_val,
            criterion, batch_size=batch_size, device=device
        )

        writer.add_scalar('Loss/Train', train_loss, epoch)
        writer.add_scalar('Loss/Val', val_loss, epoch)

        print(f"Epoch {epoch+1:3d} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} ")
    else:
        writer.add_scalar('Loss/Train', train_loss, epoch)
        print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} ")

writer.close()

Epoch   1 | Train Loss: 7.7408 
Epoch   2 | Train Loss: 7.6209 
Epoch   3 | Train Loss: 7.5753 
Epoch   4 | Train Loss: 7.5315 
Epoch   5 | Train Loss: 7.4658 | Val Loss: 0.0000 


KeyboardInterrupt: 

In [36]:
!tensorboard --logdir=runs --port=6006

2026-05-20 10:05:06.715595: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)

NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784

Serving TensorBoard on localhost; to expose to the network, use a proxy or pass --bind_all
TensorBoard 2.20.0 at http://localhost:6006/ (Press CTRL+C to quit)
^C
